# Student Lifestyle & Academic Performance Data Analysis

## Project Objective
Analyze student lifestyle and academic variables to identify patterns associated with academic performance.

### Dataset
`student_survey_data.csv`

### Main Questions
1. What does the student population look like?
2. How are study time, sleep, screen time, attendance and stress distributed?
3. Which variables are associated with GPA?
4. How does academic performance vary across selected student groups?
5. What evidence-based observations can be reported from the dataset?

> **Note:** The included CSV is a reproducible sample/synthetic dataset for demonstration. Replace it with your actual Google Forms response CSV before final submission if your form collects different variables.


In [ ]:
# Install/import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import pearsonr, ttest_ind

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")


## 1. Load the Dataset

In [ ]:
df = pd.read_csv("student_survey_data.csv")
print("Shape:", df.shape)
display(df.head())


## 2. Understand the Dataset

In [ ]:
print("Data types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isna().sum().to_frame("missing_values"))

print("\nDuplicate rows:", df.duplicated().sum())


## 3. Data Cleaning

The cleaning process:
- Remove duplicate rows.
- Convert numeric columns to numeric values.
- Fill missing numeric values with the median.
- Check that the cleaned dataset has valid ranges.


In [ ]:
numeric_cols = [
    "Age", "Study_Hours_Per_Day", "Sleep_Hours_Per_Day",
    "Screen_Time_Hours_Per_Day", "Attendance_Percent",
    "Stress_Level", "Assignments_Completed_Percent", "GPA"
]

df_clean = df.drop_duplicates().copy()

for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print("Rows after duplicate removal:", len(df_clean))
print("Remaining missing values:", df_clean.isna().sum().sum())
display(df_clean.describe().round(2))


## 4. Descriptive Statistics

In [ ]:
display(df_clean[numeric_cols].describe().T.round(2))


## 5. Categorical Analysis

In [ ]:
print("Gender distribution")
display(df_clean["Gender"].value_counts().to_frame("count"))

print("Extracurricular activity")
display(df_clean["Extracurricular_Activity"].value_counts().to_frame("count"))


## 6. Distribution Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

sns.histplot(df_clean["Study_Hours_Per_Day"], kde=True, ax=axes[0,0])
axes[0,0].set_title("Study Hours per Day")

sns.histplot(df_clean["Sleep_Hours_Per_Day"], kde=True, ax=axes[0,1])
axes[0,1].set_title("Sleep Hours per Day")

sns.histplot(df_clean["Attendance_Percent"], kde=True, ax=axes[1,0])
axes[1,0].set_title("Attendance Percentage")

sns.histplot(df_clean["GPA"], kde=True, ax=axes[1,1])
axes[1,1].set_title("GPA Distribution")

plt.tight_layout()
plt.show()


## 7. Category Comparison

In [ ]:
plt.figure(figsize=(8,5))
sns.boxplot(data=df_clean, x="Extracurricular_Activity", y="GPA")
plt.title("GPA by Extracurricular Activity")
plt.xlabel("Extracurricular Activity")
plt.ylabel("GPA")
plt.show()


## 8. Correlation Analysis

In [ ]:
corr_cols = [
    "Study_Hours_Per_Day", "Sleep_Hours_Per_Day",
    "Screen_Time_Hours_Per_Day", "Attendance_Percent",
    "Stress_Level", "Assignments_Completed_Percent", "GPA"
]

corr_matrix = df_clean[corr_cols].corr()

plt.figure(figsize=(10,7))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.show()

display(corr_matrix["GPA"].sort_values(ascending=False).to_frame("Correlation_with_GPA"))


## 9. Statistical Tests

### A. Pearson correlation
Tests the linear association between study hours and GPA.

### B. Two-sample t-test
Compares the mean GPA of students who participate in extracurricular activities with those who do not.

These tests describe relationships in this dataset; they do not establish causation.


In [ ]:
# Pearson correlation: Study hours vs GPA
r, p = pearsonr(df_clean["Study_Hours_Per_Day"], df_clean["GPA"])
print(f"Study Hours vs GPA: r = {r:.3f}, p-value = {p:.4g}")

# T-test: GPA by extracurricular activity
yes = df_clean.loc[df_clean["Extracurricular_Activity"] == "Yes", "GPA"]
no = df_clean.loc[df_clean["Extracurricular_Activity"] == "No", "GPA"]

t_stat, t_p = ttest_ind(yes, no, equal_var=False)
print(f"Extracurricular GPA comparison: t = {t_stat:.3f}, p-value = {t_p:.4g}")
print(f"Mean GPA - Yes: {yes.mean():.2f}")
print(f"Mean GPA - No: {no.mean():.2f}")


## 10. Simple Linear Regression

A simple regression model estimates GPA from study hours. This is used for interpretation and demonstration, not for making individual academic predictions.


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

X = df_clean[["Study_Hours_Per_Day"]]
y = df_clean["GPA"]

model = LinearRegression()
model.fit(X, y)

pred = model.predict(X)
print("Intercept:", round(model.intercept_, 3))
print("Coefficient for study hours:", round(model.coef_[0], 3))
print("R²:", round(r2_score(y, pred), 3))

plt.figure(figsize=(8,5))
sns.scatterplot(data=df_clean, x="Study_Hours_Per_Day", y="GPA")
plt.plot(df_clean["Study_Hours_Per_Day"], pred)
plt.title("Study Hours vs GPA with Regression Line")
plt.show()


## 11. Key Findings Template

After running the notebook on the final dataset, report:
- The size and demographic composition of the sample.
- Central tendency and spread of important numeric variables.
- The strongest correlations with GPA.
- Results of the statistical tests, including p-values.
- Important limitations such as self-reported responses, sample size, sampling method, missing data, and the difference between association and causation.

### Important
Do not claim that one factor *causes* GPA changes based only on correlation or a simple survey analysis.


## 12. Export a Clean Dataset

In [ ]:
df_clean.to_csv("cleaned_student_survey_data.csv", index=False)
print("Saved: cleaned_student_survey_data.csv")
